In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob

np.random.seed(42)
sns.set(style="whitegrid")

In [ ]:
"""
VERSION INFORMATION
"""

%load_ext watermark

%watermark --python --packages numpy,pandas,xarray,matplotlib,seaborn,scipy,sklearn,tensorflow,pytorch,jupyterlab --machine --iversions

In [ ]:
all_files = sorted(glob.glob("*.nc"))

for f in all_files:
    with xr.open_dataset(f, chunks={}) as ds:
        print(f)
        print("---Variables:", list(ds.data_vars))
        print("---Time: ", ds.valid_time.values[0], "-", ds.valid_time.values[-1])
        print("---Lat: ", float(ds.latitude.min()), "-", float(ds.latitude.max()))
        print("---Lon: ", float(ds.longitude.min()), "-", float(ds.longitude.max()))

In [ ]:
# 100m u-component of wind
u_files = sorted(glob.glob("100m_u_wind_*.nc"))
u_comb = xr.open_mfdataset(u_files, combine="by_coords")

# 100m v-component of wind
v_files = sorted(glob.glob("100m_v_wind_*.nc"))
v_comb = xr.open_mfdataset(v_files, combine="by_coords")

# ssrd
ssrd_files = sorted(glob.glob("ssrd_*.nc"))
ssrd_comb = xr.open_mfdataset(ssrd_files, combine="by_coords")

combined = xr.merge([u_comb, v_comb, ssrd_comb])
print(combined)

In [ ]:
"""
VERIFICATIONS POST-MERGE
"""

print("=== Date range ===")
print("Start:", str(combined.valid_time.values[0]))
print("End:  ", str(combined.valid_time.values[-1]))
print("Steps:", combined.valid_time.size)

print("\n=== Variables ===")
print(list(combined.data_vars))

print("\n=== Time encoding (UTC check) ===")
print(combined.valid_time.encoding.get("units", "not set"))

print("\n=== Spatial coverage ===")
print("Lat:", float(combined.latitude.min()), "-", float(combined.latitude.max()))
print("Lon:", float(combined.longitude.min()), "-", float(combined.longitude.max()))

In [ ]:
"""
MISSING VALUE CHECKS
"""

for var in combined.data_vars:
    n_missing = combined[var].isnull().sum().compute()
    total = combined[var].size
    print(f"{var}: {n_missing.values} missing out of {total} ({100 * n_missing.values / total:.4f}%)")

In [ ]:
"""
USE U AND V WIND TO ADD A NEW WIND SPEED VARAIBLE
"""

combined['wind_speed_100m_ms'] = np.sqrt(combined['u100']**2 + combined['v100']**2)

# Descriptive attributes
combined['wind_speed_100m_ms'].attrs = {
    "long_name" : "100m wind speed",
    "units" : "metres per second"
}

print(combined.data_vars)

In [ ]:
"""
AGGREGATION
"""

repd = pd.read_excel("repd.xlsx", sheet_name="REPD")

repd.columns

In [ ]:
repd = repd[["Technology Type", "Installed Capacity (MWelec)", 
             "X-coordinate", "Y-coordinate", # these are BNG eastings/northings
             "Development Status"]].copy()

In [ ]:
# Keep only operational sites
repd = repd[repd["Development Status"].str.contains("Operational", na=False)]
repd = repd.dropna(subset=["X-coordinate", "Y-coordinate", "Installed Capacity (MWelec)"])

In [ ]:
from pyproj import Transformer

transformer = Transformer.from_crs("EPSG:27700", "EPSG:4326", always_xy=True)

repd["lon"], repd["lat"] = transformer.transform(
    repd["X-coordinate"].values,
    repd["Y-coordinate"].values
)

# Clip to GB domain used for ERA5
repd = repd[
    (repd["lat"] >= 49.0) & (repd["lat"] <= 61.0) &
    (repd["lon"] >= -12.0) & (repd["lon"] <= 4.0)
]

In [ ]:
lats = combined.latitude.values # ERA5 grid
lons = combined.longitude.values

def build_capacity_weights(df, technology_keywords):
    """Bin installed capacity onto the ERA5 grid."""
    subset = df[df["Technology Type"].str.contains(
        "|".join(technology_keywords), case=False, na=False
    )]
    
    weight_grid = np.zeros((len(lats), len(lons)))
    
    for _, row in subset.iterrows():
        # Find nearest ERA5 grid point
        lat_idx = np.argmin(np.abs(lats - row["lat"]))
        lon_idx = np.argmin(np.abs(lons - row["lon"]))
        weight_grid[lat_idx, lon_idx] += row["Installed Capacity (MWelec)"]
    
    return weight_grid

wind_weights = build_capacity_weights(
    repd, ["Wind Onshore", "Wind Offshore"]
)
solar_weights = build_capacity_weights(
    repd, ["Solar Photovoltaic"]
)

# Sanity check — how much capacity was mapped?
print(f"Wind capacity mapped:  {wind_weights.sum():.0f} MW")
print(f"Solar capacity mapped: {solar_weights.sum():.0f} MW")

In [ ]:
def make_xr_weights(weight_grid, combined):
    """
    Weights = capacity (MW) × cos(latitude) for cells with capacity.
    Falls back to cos(latitude) only for cells with zero mapped capacity.
    Cosine correction accounts for ERA5 grid cell area shrinkage at higher latitudes.
    """
    cos_weights = np.cos(np.deg2rad(combined.latitude))

    da = xr.DataArray(
        weight_grid,
        dims=["latitude", "longitude"],
        coords={"latitude": combined.latitude, "longitude": combined.longitude}
    )

    # Capacity × cosine for all cells with capacity
    combined_weights = da * cos_weights

    # Cosine-only fallback for empty cells
    combined_weights = combined_weights.where(da > 0, other=cos_weights)

    return combined_weights

wind_w  = make_xr_weights(wind_weights,  combined)
solar_w = make_xr_weights(solar_weights, combined)

# Spatial aggregation
print("Computing weighted spatial means...")
wind_gb = combined["wind_speed_100m_ms"].weighted(wind_w).mean(dim=["latitude", "longitude"]).compute()
ssrd_gb = combined["ssrd"].weighted(solar_w).mean(dim=["latitude", "longitude"]).compute()
print("Done.")

In [ ]:
ig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, grid, title in zip(axes, 
                            [wind_weights, solar_weights], 
                            ["Wind capacity weights (MW)", "Solar capacity weights (MW)"]):
    im = ax.pcolormesh(lons, lats, grid, cmap="YlOrRd")
    plt.colorbar(im, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

plt.tight_layout()
plt.show()

In [ ]:
"""
BUILD DF
"""

df = pd.DataFrame({
    "timestamp_utc":       combined.valid_time.values,
    "wind_speed_100m_ms":  wind_gb.values,
    "ssrd_j_m2":           ssrd_gb.values,
})

df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
df = df.sort_values("timestamp_utc").reset_index(drop=True)

# Clip small negative ssrd values (ERA5 spectral artefact)
ssrd_min_before_clip = df["ssrd_j_m2"].min()
df["ssrd_j_m2"] = df["ssrd_j_m2"].clip(lower=0)

print(df.head())

In [ ]:
"""
CHECKS
"""

print("=== Duplicate timestamps ===")
dupes = df["timestamp_utc"].duplicated().sum()
print(f"None" if dupes == 0 else f"{dupes} duplicates")

print("\n=== Full hourly coverage 2010–2024 ===")
expected = pd.date_range("2010-01-01", "2024-12-31 23:00", freq="h", tz="UTC")
missing_ts = expected.difference(df["timestamp_utc"])
print(f"No missing hours" if len(missing_ts) == 0 else f"{len(missing_ts)} missing hours")

print("\n=== Missing value counts ===")
wind_missing = df["wind_speed_100m_ms"].isna().sum()
ssrd_missing = df["ssrd_j_m2"].isna().sum()
print(f"wind_speed_100m_ms: {wind_missing} missing")
print(f"ssrd_j_m2: {ssrd_missing} missing")

print("\n=== Sample stats (unit sense-check) ===")
print(df[["wind_speed_100m_ms", "ssrd_j_m2"]].describe().round(3))

In [ ]:
"""
SAVE TO PARQUET
"""

df.to_parquet("era5_resource_hourly_gb_2010_2024.parquet", index=False)

# Verify
df_check = pd.read_parquet("era5_resource_hourly_gb_2010_2024.parquet")
print(df_check.shape) # Expect (131496, 3)
print(df_check.dtypes)

In [ ]:
readme = f"""
# ERA5 GB Hourly Resource Data — 2010–2024

## File
era5_resource_hourly_gb_2010_2024.parquet

## Columns
| Column               | Units  | Notes |
|----------------------|--------|-------|
| timestamp_utc        | —      | Hourly, UTC, 2010-01-01 00:00 - 2024-12-31 23:00 |
| wind_speed_100m_ms   | m/s    | Derived from ERA5 u100/v100: sqrt(u^2 + v^2) |
| ssrd_j_m2            | J/m^2  | ERA5 native units (hourly accumulation). To convert to mean W/m^2: divide by 3600 |

## Spatial aggregation
Capacity-weighted spatial mean with cosine-latitude area correction.
Each grid cell weight = installed capacity (MW) × cos(latitude).
Grid cells with no mapped capacity fall back to cosine-latitude weighting only.

    Wind:  weighted by operational onshore + offshore wind capacity (REPD)
    Solar: weighted by operational solar PV capacity (REPD)
    Domain: Lat 49.0–61.0 | Lon -12.0–4.0 | 0.25° resolution

Source: Renewable Energy Planning Database (REPD), DESNZ.

Note: REPD capacity weights reflect the fleet at time of download (snapshot).
A 2010 timestep receives the same weights as a 2024 timestep. This is a known
and accepted simplification in GB energy system modelling.

## Data cleaning
ssrd_j_m2: small negative values clipped to zero (ERA5 spectral artefact).
    Min value before clip: {ssrd_min_before_clip:.3f} J/m²

## Quality checks
    Duplicate timestamps: {dupes}
    Missing hours 2010–2024: {len(missing_ts)}
    Missing values (wind): {wind_missing}
    Missing values (ssrd): {ssrd_missing}

## Source data
ERA5 reanalysis, ECMWF via Copernicus CDS.
All timestamps are UTC (encoding: hours since 1900-01-01 00:00:00.0 UTC).
""".strip()

with open("README.md", "w") as f:
    f.write(readme)

print(readme)